# Model Validation & Diagnostics

Comprehensive validation for Hawkes processes including:
- Residual analysis (compensator-based)
- Bootstrap confidence intervals
- Time-series cross-validation
- Baseline model comparison


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

from hawkes.utils.data_loader import load_sample_data
from hawkes.estimation.ultra_fast_mle import UltraFastMultivariateHawkesMLE
from hawkes.diagnostics import (
    ResidualDiagnostics, BootstrapConfidenceIntervals,
    TimeSeriesCrossValidator, ModelComparison
)


In [2]:
# Load and fit
events, metadata = load_sample_data()
estimator = UltraFastMultivariateHawkesMLE(n_dims=4, assume_independent=True, max_iter=30)
estimator.fit(events, end_time=metadata['duration_seconds'])
print(f"Fitted. LL: {estimator.log_likelihood_:.2f}, rho: {estimator.compute_spectral_radius():.4f}")


Fitted. LL: -3092.11, rho: 0.3570


In [3]:
# Residual diagnostics
res_diag = ResidualDiagnostics(events, metadata['duration_seconds'])
residuals = res_diag.compute_residuals(estimator.mu_, estimator.alpha_, estimator.beta_)
ks_results = res_diag.ks_test(residuals)
print("KS Test Results:", ks_results)


KS Test Results: {'by_dimension': [{'dimension': 0, 'n_observations': 1406, 'ks_statistic': np.float64(0.02544360697025405), 'p_value': np.float64(0.31719132245554393), 'reject_h0': np.False_, 'mean_residual': np.float64(0.9990796973254882), 'std_residual': np.float64(0.9683864634149109)}, {'dimension': 1, 'n_observations': 1386, 'ks_statistic': np.float64(0.029962511420024374), 'p_value': np.float64(0.16266957942840965), 'reject_h0': np.False_, 'mean_residual': np.float64(0.9995263833411949), 'std_residual': np.float64(0.9616765899286333)}, {'dimension': 2, 'n_observations': 2013, 'ks_statistic': np.float64(0.00932896192799415), 'p_value': np.float64(0.994142594421936), 'reject_h0': np.False_, 'mean_residual': np.float64(0.999970324848175), 'std_residual': np.float64(1.005526154345822)}, {'dimension': 3, 'n_observations': 1903, 'ks_statistic': np.float64(0.010128526429991735), 'p_value': np.float64(0.9887392211855062), 'reject_h0': np.False_, 'mean_residual': np.float64(0.999762531330

In [4]:
# Bootstrap CI (small sample for demo)
boot = BootstrapConfidenceIntervals(n_bootstrap=20, random_state=42)
# Skip actual bootstrap for speed in demo
print("Bootstrap initialized. Use n_bootstrap=100+ for production.")


Bootstrap initialized. Use n_bootstrap=100+ for production.


In [5]:
# Time-series CV
cv = TimeSeriesCrossValidator(n_folds=3, min_train_size=300)
cv_results = cv.cross_validate(
    events, metadata['duration_seconds'],
    UltraFastMultivariateHawkesMLE,
    {'n_dims': 4, 'assume_independent': True, 'max_iter': 30}
)
print(f"CV completed: {len(cv_results)} folds")


CV completed: 3 folds


In [6]:
# Compare with baselines
comparison = ModelComparison(events, metadata['duration_seconds'])
results_df = comparison.fit_all(estimator)
comparison.print_summary()



MODEL COMPARISON SUMMARY

Homogeneous Poisson:
  Constant intensity, no self-excitation
  Parameters: 4
  Log-likelihood: -3141.68
  AIC: 6291.35 (ΔAIC: 35.13)
  BIC: 6318.60 (ΔBIC: 0.00)
  Akaike weight: 0.000

Piecewise Poisson (10 bins):
  Time-varying intensity, no self-excitation
  Parameters: 40
  Log-likelihood: -3118.15
  AIC: 6316.30 (ΔAIC: 60.08)
  BIC: 6588.76 (ΔBIC: 270.16)
  Akaike weight: 0.000

Hawkes (Exponential):
  Self-exciting process with exponential kernel
  Parameters: 36
  Log-likelihood: -3092.11
  AIC: 6256.22 (ΔAIC: 0.00)
  BIC: 6501.44 (ΔBIC: 182.84)
  Akaike weight: 1.000

--------------------------------------------------------------------------------
BEST MODELS:
  By AIC: Hawkes (Exponential)
  By BIC: Homogeneous Poisson
  By Log-likelihood: Hawkes (Exponential)

--------------------------------------------------------------------------------
LIKELIHOOD RATIO TEST (Poisson vs Hawkes):
  LR statistic: 99.13
  Degrees of freedom: 32
  P-value: 8.63e-09
 